# Text-to-Image Synthesis: BERT + Conditional GAN, evaluated with CLIP

Pipeline (PyTorch):
1. **BERT** (frozen) encodes each caption; a learned layer projects it to a **256-d text embedding**.
2. **Generator**: noise + text embedding -> ResNet upsampling with conditional BatchNorm -> 32x32 image.
3. **Discriminator**: judges **real vs fake** and **whether the image matches the caption** (projection conditioning + mismatched-caption negatives).
4. **Stabilisation**: one-sided **label smoothing**, decaying **noise injection** (instance noise) on discriminator inputs, **different learning rates** for G and D (TTUR), spectral norm, EMA generator.
5. **Extra training aids**: DiffAugment (for the small 10k dataset), LR decay, EMA warm-up, best-checkpoint selection by CLIP score.
6. **Evaluation**: **CLIP similarity** between generated images and their captions, plus CLIP zero-shot accuracy.

Data: **10,000 image-text pairs** (1,000 CIFAR-10 images per class, each paired with one caption).

In [ ]:
!pip -q install transformers

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, copy, time, os, random
import torchvision
from torch.nn.utils import spectral_norm as SN
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

EPOCHS = 300           # 10k images -> 156 iters/epoch
BS = 64
ZDIM = 128
TDIM = 256             # text embedding size (BERT 768 -> 256)
CH = 128
LR_G, LR_D = 2e-4, 4e-4        # different learning rates (TTUR)
REAL_LABEL = 0.9               # one-sided label smoothing
NOISE_START = 0.1              # instance-noise std, decays to 0
PER_CLASS = 1000               # 10 classes x 1000 = 10,000 image-text pairs
CKPT = "gan_ckpt.pt"

NAMES = ["an airplane", "a car", "a bird", "a cat", "a deer", "a dog", "a frog", "a horse", "a ship", "a truck"]
TEMPLATES = ["a photo of {}", "a picture of {}", "{} in the scene", "a small image of {}",
             "a clear photo showing {}", "this is {}", "{}", "a close-up of {}",
             "an image of {}", "a clear picture of {}", "a good photo of {}", "a centered photo of {}"]
CAPTIONS = [t.format(n) for n in NAMES for t in TEMPLATES]   # caption id = class * NT + template
NT = len(TEMPLATES)

device: cpu


## 1. BERT caption embeddings (frozen)

In [ ]:
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased").to(device).eval()

@torch.no_grad()
def encode(texts):
    b = tok(texts, padding=True, truncation=True, max_length=32, return_tensors="pt").to(device)
    h = bert(**b).last_hidden_state
    m = b["attention_mask"].unsqueeze(-1).float()
    return (h * m).sum(1) / m.sum(1)          # mean-pooled 768-d

EMB = encode(CAPTIONS)                         # [80, 768]
print("caption embeddings:", EMB.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


caption embeddings: torch.Size([120, 768])


## 2. Dataset: 10,000 image-text pairs

In [ ]:
raw = torchvision.datasets.CIFAR10("data", train=True, download=True)
import numpy as np
targets = np.array(raw.targets)
g = np.random.RandomState(SEED)
idx = np.concatenate([g.choice(np.where(targets == c)[0], PER_CLASS, replace=False) for c in range(10)])
X = torch.from_numpy(raw.data[idx]).permute(0, 3, 1, 2).float().div(127.5).sub(1).to(device)   # [-1,1]
Y = torch.from_numpy(targets[idx]).to(device)
CID = Y * NT + torch.from_numpy(g.randint(0, NT, len(idx))).to(device)    # one fixed caption per image
N = len(X)
print(N, "image-text pairs. Example:", CAPTIONS[CID[0].item()])

100%|██████████| 170M/170M [05:07<00:00, 555kB/s]


10000 image-text pairs. Example: an airplane


## 3. Generator and Discriminator

In [ ]:
class CBN(nn.Module):
    def __init__(s, c, cd):
        super().__init__()
        s.bn = nn.BatchNorm2d(c, affine=False)
        s.g = nn.Linear(cd, c); s.b = nn.Linear(cd, c)
    def forward(s, x, c):
        return s.bn(x) * (1 + s.g(c))[:, :, None, None] + s.b(c)[:, :, None, None]

class GBlock(nn.Module):
    def __init__(s, i, o, cd):
        super().__init__()
        s.n1 = CBN(i, cd); s.c1 = nn.Conv2d(i, o, 3, padding=1)
        s.n2 = CBN(o, cd); s.c2 = nn.Conv2d(o, o, 3, padding=1)
        s.sk = nn.Conv2d(i, o, 1)
    def forward(s, x, c):
        h = s.c1(F.interpolate(F.relu(s.n1(x, c)), scale_factor=2))
        h = s.c2(F.relu(s.n2(h, c)))
        return h + s.sk(F.interpolate(x, scale_factor=2))

class Generator(nn.Module):
    def __init__(s):
        super().__init__()
        s.txt = nn.Sequential(nn.Linear(768, 512), nn.ReLU(), nn.Linear(512, TDIM))   # BERT -> 256-d
        cd = ZDIM + TDIM
        s.fc = nn.Linear(cd, 4 * 4 * CH * 2)
        s.b1 = GBlock(CH * 2, CH * 2, cd)
        s.b2 = GBlock(CH * 2, CH, cd)
        s.b3 = GBlock(CH, CH, cd)
        s.bn = nn.BatchNorm2d(CH)
        s.out = nn.Conv2d(CH, 3, 3, padding=1)
    def forward(s, z, e):
        c = torch.cat([z, s.txt(e)], 1)
        h = s.fc(c).view(-1, CH * 2, 4, 4)
        h = s.b3(s.b2(s.b1(h, c), c), c)
        return torch.tanh(s.out(F.relu(s.bn(h))))

class DBlock(nn.Module):
    def __init__(s, i, o, down=True, first=False):
        super().__init__()
        s.c1 = SN(nn.Conv2d(i, o, 3, padding=1)); s.c2 = SN(nn.Conv2d(o, o, 3, padding=1))
        s.sk = SN(nn.Conv2d(i, o, 1)); s.down = down; s.first = first
    def forward(s, x):
        h = x if s.first else F.relu(x)
        h = s.c2(F.relu(s.c1(h)))
        sk = s.sk(x)
        if s.down:
            h = F.avg_pool2d(h, 2); sk = F.avg_pool2d(sk, 2)
        return h + sk

class Discriminator(nn.Module):
    def __init__(s):
        super().__init__()
        s.blocks = nn.Sequential(DBlock(3, CH, True, True), DBlock(CH, CH, True), DBlock(CH, CH, False), DBlock(CH, CH, False))
        s.fc = SN(nn.Linear(CH, 1))                                   # real / fake
        s.txt = nn.Sequential(SN(nn.Linear(768, 512)), nn.ReLU(), SN(nn.Linear(512, CH)))   # text projection
    def forward(s, x, e):
        h = F.relu(s.blocks(x)).sum([2, 3])
        return s.fc(h).squeeze(1) + (s.txt(e) * h).sum(1)             # realism + caption match

G = Generator().to(device); D = Discriminator().to(device)
G_ema = copy.deepcopy(G).eval()
print("G params: %.1fM  D params: %.1fM" % (sum(p.numel() for p in G.parameters())/1e6, sum(p.numel() for p in D.parameters())/1e6))

G params: 5.0M  D params: 1.5M


## 4. Helpers

In [ ]:
def show(imgs, titles, cols=5, scale=2.2):
    imgs = ((imgs.cpu() + 1) / 2).clamp(0, 1)
    rows = math.ceil(len(imgs) / cols)
    fig, ax = plt.subplots(rows, cols, figsize=(cols * scale, rows * (scale + 0.4)))
    for a in ax.flat: a.axis("off")
    for a, im, t in zip(ax.flat, imgs, titles):
        a.imshow(im.permute(1, 2, 0), interpolation="bicubic"); a.set_title(t, fontsize=8)
    plt.tight_layout(); plt.show()

@torch.no_grad()
def generate(caps, net=None, trunc=1.0):
    net = net or G_ema
    net.eval()
    return net(torch.randn(len(caps), ZDIM, device=device) * trunc, encode(caps))

def preview():
    caps = [f"a photo of {n}" for n in NAMES]
    show(generate(caps, trunc=0.8), caps)

def bce(logits, target):
    return F.binary_cross_entropy_with_logits(logits, torch.full_like(logits, target))

# ---- DiffAugment: differentiable augmentation applied to ALL discriminator inputs ----
def _color(x):
    B = x.size(0)
    x = x + (torch.rand(B, 1, 1, 1, device=x.device) - 0.5)
    m = x.mean(1, keepdim=True)
    x = (x - m) * (torch.rand(B, 1, 1, 1, device=x.device) * 2) + m
    m = x.mean([1, 2, 3], keepdim=True)
    return (x - m) * (torch.rand(B, 1, 1, 1, device=x.device) + 0.5) + m

def _translate(x, ratio=0.125):
    B, C, H, W = x.shape
    sh = int(H * ratio + 0.5)
    tx = torch.randint(-sh, sh + 1, (B, 1, 1), device=x.device)
    ty = torch.randint(-sh, sh + 1, (B, 1, 1), device=x.device)
    gb, gx, gy = torch.meshgrid(torch.arange(B, device=x.device), torch.arange(H, device=x.device),
                                torch.arange(W, device=x.device), indexing="ij")
    gx = torch.clamp(gx + tx + 1, 0, H + 1); gy = torch.clamp(gy + ty + 1, 0, W + 1)
    xp = F.pad(x, [1, 1, 1, 1])
    return xp.permute(0, 2, 3, 1).contiguous()[gb, gx, gy].permute(0, 3, 1, 2)

def _cutout(x, ratio=0.5):
    B, C, H, W = x.shape
    cs = int(H * ratio + 0.5)
    ox = torch.randint(0, H + (1 - cs % 2), (B, 1, 1), device=x.device)
    oy = torch.randint(0, W + (1 - cs % 2), (B, 1, 1), device=x.device)
    gb, gx, gy = torch.meshgrid(torch.arange(B, device=x.device), torch.arange(cs, device=x.device),
                                torch.arange(cs, device=x.device), indexing="ij")
    gx = torch.clamp(gx + ox - cs // 2, 0, H - 1); gy = torch.clamp(gy + oy - cs // 2, 0, W - 1)
    mask = torch.ones(B, H, W, device=x.device); mask[gb, gx, gy] = 0
    return x * mask.unsqueeze(1)

def aug(x):
    return _cutout(_translate(_color(x)))

## 5a. CLIP helpers + evaluation functions
Measures how well generated images align with their captions:
- **CLIP similarity**: cosine similarity between image and its caption (higher = better alignment)
- **Mismatched baseline**: same images vs. captions of *other* classes (should be lower)
- **Real images**: reference upper bound
- **CLIP zero-shot accuracy**: does CLIP pick the right class for the generated image?
Also used *during training* to keep the best generator checkpoint. The table at the end is the final evaluation.

In [ ]:
from transformers import CLIPModel, CLIPTokenizer
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
ctok = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
CM = torch.tensor([0.4815, 0.4578, 0.4082], device=device).view(1, 3, 1, 1)
CS = torch.tensor([0.2686, 0.2613, 0.2758], device=device).view(1, 3, 1, 1)

def _feat(o):
    return o if torch.is_tensor(o) else o.pooler_output

@torch.no_grad()
def clip_img(x):
    x = ((x + 1) / 2).clamp(0, 1)
    x = F.interpolate(x, size=224, mode="bicubic", align_corners=False).clamp(0, 1)
    return F.normalize(_feat(clip.get_image_features(pixel_values=(x - CM) / CS)), dim=-1)

@torch.no_grad()
def clip_txt(texts):
    t = ctok(texts, padding=True, return_tensors="pt").to(device)
    return F.normalize(_feat(clip.get_text_features(**t)), dim=-1)

PROMPTS = [f"a photo of {n}" for n in NAMES]
T_FEAT = clip_txt(PROMPTS)                    # [10, 512]

def evaluate(images_by_class):
    match, mism, correct, total = [], [], 0, 0
    for c, imgs in enumerate(images_by_class):
        sims = clip_img(imgs) @ T_FEAT.T       # [n, 10]
        match.append(sims[:, c].mean().item())
        mism.append(sims[:, [k for k in range(10) if k != c]].mean().item())
        correct += (sims.argmax(1) == c).sum().item(); total += len(imgs)
    return sum(match) / 10, sum(mism) / 10, correct / total

def quick_eval(n=30):
    return evaluate([generate([PROMPTS[c]] * n, trunc=0.8) for c in range(10)])

@torch.no_grad()
def generate_best(caption, k=32, n=1, trunc=0.8):
    # sample k candidates, keep the n with highest CLIP image-text similarity
    imgs = generate([caption] * k, trunc=trunc)
    sc = (clip_img(imgs) @ clip_txt([caption]).T).squeeze(1)
    return imgs[sc.topk(n).indices], sc.topk(n).values

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

## 5b. Train (auto-resumes from `gan_ckpt.pt`)
- **Label smoothing:** real target = 0.9
- **Noise injection:** Gaussian noise on D inputs, std decays linearly to 0 over the first 70% of training
- **Learning rates:** G = 2e-4, D = 4e-4
- **D loss:** real+matching caption -> real; fake -> fake; real image + wrong caption -> fake
- **Extras:** DiffAugment on all D inputs, LR decay over the last 40%, EMA warm-up, best checkpoint kept by CLIP score

In [ ]:
optG = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(0.0, 0.9))
optD = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(0.0, 0.9))
start, step = 0, 0
best_score, best_state = -1.0, None
if os.path.exists(CKPT):
    s = torch.load(CKPT, map_location=device)
    G.load_state_dict(s["G"]); D.load_state_dict(s["D"]); G_ema.load_state_dict(s["G_ema"])
    optG.load_state_dict(s["optG"]); optD.load_state_dict(s["optD"]); start = s["epoch"]
    step, best_score, best_state = s.get("step", 0), s.get("best_score", -1.0), s.get("best_state")
    print("resumed from epoch", start)

history = []
for epoch in range(start, EPOCHS):
    G.train(); D.train(); t0 = time.time(); dl = gl = 0; nb = 0
    sigma = NOISE_START * max(0.0, 1 - epoch / (0.7 * EPOCHS))
    f = 1.0 if epoch < 0.6 * EPOCHS else max(0.05, 1 - (epoch - 0.6 * EPOCHS) / (0.4 * EPOCHS))
    for gp in optG.param_groups: gp["lr"] = LR_G * f
    for gp in optD.param_groups: gp["lr"] = LR_D * f
    perm = torch.randperm(N, device=device)
    for i in range(0, N - BS + 1, BS):
        b = perm[i:i + BS]
        x, y, cid = X[b], Y[b], CID[b]
        flip = torch.rand(BS, 1, 1, 1, device=device) < 0.5
        x = torch.where(flip, x.flip(3), x)
        e = EMB[cid]
        wrong = EMB[((y + torch.randint(1, 10, y.shape, device=device)) % 10) * NT + torch.randint(0, NT, y.shape, device=device)]
        z = torch.randn(BS, ZDIM, device=device)
        noisy = lambda t: t + sigma * torch.randn_like(t)

        # --- discriminator ---
        with torch.no_grad():
            fake = G(z, e)
        lossD = bce(D(noisy(aug(x)), e), REAL_LABEL) + 0.5 * (bce(D(noisy(aug(fake)), e), 0.0) + bce(D(noisy(aug(x)), wrong), 0.0))
        optD.zero_grad(set_to_none=True); lossD.backward(); optD.step()

        # --- generator ---
        z = torch.randn(BS, ZDIM, device=device)
        lossG = bce(D(noisy(aug(G(z, e))), e), 1.0)
        optG.zero_grad(set_to_none=True); lossG.backward(); optG.step()

        step += 1
        decay = min(0.999, (1 + step) / (10 + step))      # EMA warm-up
        with torch.no_grad():
            for pe, pg in zip(G_ema.parameters(), G.parameters()): pe.lerp_(pg, 1 - decay)
            for be, bg in zip(G_ema.buffers(), G.buffers()): be.copy_(bg)
        dl += lossD.item(); gl += lossG.item(); nb += 1
    history.append((dl / nb, gl / nb))
    print(f"epoch {epoch+1}/{EPOCHS}  D {dl/nb:.3f}  G {gl/nb:.3f}  noise {sigma:.3f}  {time.time()-t0:.0f}s")
    if (epoch + 1) >= EPOCHS // 3 and (epoch + 1) % 25 == 0:
        m, mm, a = quick_eval()
        score = a + m
        print(f"   CLIP check: sim {m:.3f}  mismatch {mm:.3f}  zero-shot acc {a:.1%}")
        if score > best_score:
            best_score, best_state = score, copy.deepcopy(G_ema.state_dict())
            print("   -> new best generator")
    if (epoch + 1) % 10 == 0:
        torch.save({"G": G.state_dict(), "D": D.state_dict(), "G_ema": G_ema.state_dict(),
                    "optG": optG.state_dict(), "optD": optD.state_dict(), "epoch": epoch + 1,
                    "step": step, "best_score": best_score, "best_state": best_state}, CKPT)
    if (epoch + 1) % 20 == 0:
        preview()

epoch 1/300  D 2.117  G 1.475  noise 0.100  2788s
epoch 2/300  D 1.391  G 0.788  noise 0.100  2695s


## 6. Try your own captions (CLIP picks the best of 32 candidates per caption)
Name one of: airplane, car, bird, cat, deer, dog, frog, horse, ship, truck.

In [ ]:
my_captions = [
    "a photo of a horse standing in a field",
    "a small bird",
    "a picture of a ship on the sea",
    "a clear photo showing a truck",
    "a cat",
    "an airplane flying in the sky",
    "a green frog",
    "a dog on grass",
]
if best_state is not None:
    G_ema.load_state_dict(best_state)
picked = torch.cat([generate_best(c, k=32)[0] for c in my_captions])
show(picked, my_captions, cols=4, scale=3)

## 7. Final CLIP evaluation (loads the best checkpoint, compares against real images)

In [ ]:
NEVAL = 100
if best_state is not None:
    G_ema.load_state_dict(best_state)
    print(f"Loaded best checkpoint (selection score {best_score:.4f})")

@torch.no_grad()
def reranked(c, n=NEVAL, pool=2):
    imgs = generate([PROMPTS[c]] * (n * pool), trunc=0.8)
    sc = (clip_img(imgs) @ T_FEAT[c:c + 1].T).squeeze(1)
    return imgs[sc.topk(n).indices]

real = [X[Y == c][:NEVAL] for c in range(10)]
rows = {
    "Generated (raw)": evaluate([torch.cat([generate([PROMPTS[c]] * 50), generate([PROMPTS[c]] * 50)]) for c in range(10)]),
    "Generated (trunc 0.8)": evaluate([torch.cat([generate([PROMPTS[c]] * 50, trunc=0.8), generate([PROMPTS[c]] * 50, trunc=0.8)]) for c in range(10)]),
    "Gen + CLIP rerank*": evaluate([reranked(c) for c in range(10)]),
    "Real images": evaluate(real),
}
print(f"{'':24s}{'CLIP sim (match)':>18s}{'CLIP sim (mismatch)':>21s}{'zero-shot acc':>15s}")
for k, (m, mm, a) in rows.items():
    print(f"{k:24s}{m:18.4f}{mm:21.4f}{a:15.1%}")
print("*reranking selects images using CLIP itself, so it is inflated; report the raw / trunc rows as the model's quality.")

## 8. Save to your computer

In [ ]:
try:
    from google.colab import files
    files.download(CKPT)
except ImportError:
    print("Not on Colab - gan_ckpt.pt is saved in this folder.")